# SIFT Landmark Recognition and Verification Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Afnan-makstrix/sift-landmark-recognition-pipeline/blob/main/SIFT_Landmark_Recognition_Pipeline.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-teal?logo=github)](https://github.com/Afnan-makstrix/sift-landmark-recognition-pipeline)
[![Student](https://img.shields.io/badge/Student-Afnan%20Khan-emerald)](https://github.com/Afnan-makstrix)
[![Course](https://img.shields.io/badge/Course-Computer%20Vision-blueviolet)](https://github.com/Afnan-makstrix)

**Student Name:** Afnan Khan ([@Afnan-makstrix](https://github.com/Afnan-makstrix))  
**Department:** Computer Science / Artificial Intelligence  
**Project:** Automated Tourism Information & Landmark Identification System  

---

## 1. Project Overview

This project implements an end-to-end Computer Vision pipeline to recognize famous architectural landmarks from tourist query photographs using the **Scale-Invariant Feature Transform (SIFT)**, **Brute-Force KNN Feature Matching**, **Lowe's Ratio Testing**, and **RANSAC Homography Localization**.

### Architectural Stages
1. **Task 1: Reference Database Ingestion** - Storing multi-angle reference imagery for 5 world landmarks + negative sample.
2. **Task 2: SIFT Feature Extraction** - Scale-space extrema detection, subpixel refinement, and 128-D descriptor computation.
3. **Task 3: Query Image Processing** - Dynamic feature extraction for unlabelled incoming tourist imagery.
4. **Task 4: KNN Matching & Rejection Filter** - Euclidean $L_2$ distance matching with $k=2$ and Lowe's 0.75 ratio test.
5. **Task 5: Landmark Localization** - RANSAC geometric verification and projective boundary polygon mapping.
6. **Task 6: Robustness Stress-Testing** - Benchmarking under rotation, scaling, darkness, perspective shifts, and non-database rejection.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sift_landmark_system import LandmarkRecognitionPipeline

print(f"OpenCV version: {cv2.__version__}")
pipeline = LandmarkRecognitionPipeline()
database = pipeline.build_database('data/landmarks')
print(f"Loaded database with {len(database)} landmarks:")
for k, v in database.items():
    print(f" - {k}: {len(v)} reference views")

## 2. Landmark Reference Database Visual Showcase

Reference gallery containing the 5 target landmarks:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (lm, refs) in zip(axes, sorted(database.items())):
    rgb = cv2.cvtColor(refs[0]['bgr'], cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    ax.set_title(lm.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. SIFT Scale-Space Keypoints

Rich keypoint circles indicating localized feature scale and gradient orientation:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (lm, refs) in zip(axes, sorted(database.items())):
    ref = refs[0]
    vis = cv2.drawKeypoints(ref['bgr'], ref['kps'][:350], None, color=(0, 255, 128),
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{lm}\n{len(ref['kps'])} SIFT KPs", fontsize=10, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Feature Matching, Verification & Geometric Localization

In [ ]:
# Query evaluation on rotated Eiffel Tower
eiffel_ref = database['eiffel_tower'][0]['bgr']
h, w = eiffel_ref.shape[:2]
M = cv2.getRotationMatrix2D((w/2, h/2), 45, 0.85)
q_rot = cv2.warpAffine(eiffel_ref, M, (w, h), borderMode=cv2.BORDER_REFLECT)

rec = pipeline.recognize(q_rot)
print(rec['decision_message'])

if rec['is_recognized']:
    top = rec['top_score']
    loc = pipeline.localize(q_rot, rec['query_kps'], top['ref']['bgr'], top['ref']['kps'], top['matches'])
    print(f"Localization Status: {loc['msg']} | Inliers: {loc['inliers']}")
    plt.figure(figsize=(7, 7))
    plt.imshow(cv2.cvtColor(loc['image'], cv2.COLOR_BGR2RGB))
    plt.title("Localized Landmark Polygon", fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.show()

## 5. Performance Evaluation & Environmental Stress-Testing

Comprehensive evaluation table across all 6 test scenarios:

| Condition | Ground Truth | Predicted Landmark | Query KPs | Matches | Inliers | Decision | Latency |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Original Image** | `eiffel_tower` | `eiffel_tower` | `2,946` | `2,946` | `2,946` | **RECOGNIZED** | `909.0 ms` |
| **Rotated Image (45 deg)** | `eiffel_tower` | `eiffel_tower` | `2,517` | `1,234` | `1,227` | **RECOGNIZED** | `808.2 ms` |
| **Resized Image (0.5x Scale)** | `eiffel_tower` | `eiffel_tower` | `672` | `509` | `508` | **RECOGNIZED** | `263.2 ms` |
| **Reduced Brightness (0.35x)** | `eiffel_tower` | `eiffel_tower` | `760` | `718` | `717` | **RECOGNIZED** | `315.9 ms` |
| **Different Viewpoint** | `eiffel_tower` | `eiffel_tower` | `1,020` | `1,020` | `1,020` | **RECOGNIZED** | `336.2 ms` |
| **Non-Database Landmark** | `unknown` | `Unknown / Rejected` | `1,984` | `22` | `0` | **REJECTED** | `644.3 ms` |

### Robustness Evaluation Gallery
![Robustness Evaluation](figures/task6_robustness_evaluation.png)

### Latency & Match Telemetry
![Performance Metrics](figures/task6_performance_metrics.png)

## 6. Discussion and System Limitations

1. **SIFT Invariance Reliability:** Features exhibited near-perfect invariance to rotation (1,234 matches, 1,227 inliers) and down-scaling (509 matches, 508 inliers).
2. **Photometric Stability:** Low brightness (0.35x) did not compromise classification because normalized gradient orientation histograms are intrinsically invariant to linear intensity shifts.
3. **False-Positive Prevention:** Non-database queries (Big Ben) yielded only 22 spurious matches and 0 RANSAC inliers, correctly triggering system rejection.
4. **Scalability Considerations:** While Brute-Force matching is effective for small collections, real-world tourism databases with thousands of monuments benefit from KD-tree indexing (FLANN) or vector quantization (Bag-of-Words).